<a href="https://colab.research.google.com/github/ahmedlayouni2001/WiseVision/blob/main/annotations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== CRK seller/client dataset builder + classifier =====

import importlib, json, os, random, shutil, subprocess, sys
from collections import defaultdict
from pathlib import Path

def _ensure(pkg, mod=None):
    try:
        importlib.import_module(mod or pkg)
    except ImportError:
        print(f"[SETUP] installing {pkg} ...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
        importlib.invalidate_caches()

_ensure("ultralytics")
_ensure("pyyaml", "yaml")

import cv2, numpy as np, torch

# ---------------- CONFIG ----------------
VIDEO_PATH = "/kaggle/input/datasets/maramkouki/trainset/part1_00-00_40-00.mp4"
OUT_DIR    = "/kaggle/working/crk_dataset"
RUN_EXTRACT, RUN_TRAIN = True, True

SKIP = 25            # 25 = fast pass, 5 = full dataset
MAX_MINUTES = 0      # 0 = whole video

MODEL_NAME, DETECT_CONF, PERSON_CLASS_ID = "yolo11m.pt", 0.35, 0
TRACK_HIGH_THRESH, TRACK_LOW_THRESH, NEW_TRACK_THRESH = 0.4, 0.1, 0.2
TRACK_BUFFER, MATCH_THRESH = 180, 0.8
PROXIMITY_THRESH, APPEARANCE_THRESH = 0.8, 0.3

UNIFORM_BAND_LO = np.array([18, 0, 51])
UNIFORM_BAND_HI = np.array([49, 54, 240])
WHITE_S_MAX, WHITE_V_MIN = 80, 85

SELLER_COLOR_MIN, SELLER_WHITE_MIN = 0.55, 0.50
CLIENT_COLOR_MAX, CLIENT_WHITE_MAX = 0.08, 0.15

MIN_BOX_H, MIN_BOX_W = 90, 35
MIN_ASPECT, MAX_ASPECT, CROP_PAD = 1.3, 5.0, 0.04
CROP_SIZE = (128, 256)

MAX_PER_TRACK, MIN_FRAME_GAP = 25, 15
MAX_PER_CLASS, MAX_REVIEW = 4000, 1500

EPOCHS, BATCH_SIZE, LR, VAL_FRAC, SEED = 12, 64, 3e-4, 0.20, 42

ZONE_REF_W, ZONE_REF_H = 1920, 1080
MIRROR_ZONES = [
    [(36,125),(195,59),(419,635),(168,745),(34,131)],
    [(1732,3),(1888,4),(1788,493),(1654,448),(1730,8)],
    [(904,3),(1124,4),(1155,61),(1161,240),(929,276),(902,10)],
]
SELLER_ZONES = [
    [(198,851),(478,696),(1001,1001),(1086,1068),(325,1068),(196,854)],
    [(964,887),(1416,893),(1428,1068),(894,1069),(959,888)],
    [(161,780),(471,674),(494,704),(201,847),(161,786)],
    [(1008,255),(992,140),(1152,146),(1159,247),(1008,255)],
    [(695,666),(592,565),(528,381),(705,323),(732,387),(698,665)],
]
IGNORE_ZONES = [
    [(1789,663),(1838,680),(1819,768),(1769,750),(1788,665)],
    [(404,753),(649,1068),(914,941),(588,651),(408,750)],
    [(502,674),(471,661),(316,740),(329,794),(501,681)],
]
SELLER_ZONE_FRAC = 0.70
# ----------------------------------------

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
HALF = torch.cuda.is_available()
CLASSES = ["client", "seller"]
_MIRROR_POLYS, _SELLER_POLYS, _IGNORE_POLYS = [], [], []
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


def build_polys(width, height):
    global _MIRROR_POLYS, _SELLER_POLYS, _IGNORE_POLYS
    sx, sy = width / float(ZONE_REF_W), height / float(ZONE_REF_H)
    def conv(zones):
        out = []
        for z in zones:
            if len(z) < 3: continue
            out.append(np.array([(int(round(x*sx)), int(round(y*sy))) for x, y in z], np.int32))
        return out
    _MIRROR_POLYS, _SELLER_POLYS, _IGNORE_POLYS = conv(MIRROR_ZONES), conv(SELLER_ZONES), conv(IGNORE_ZONES)
    if abs(sx-1) > 1e-6 or abs(sy-1) > 1e-6:
        print(f"[ZONES] rescaled by ({sx:.3f}, {sy:.3f}) for {width}x{height}")


def find_video(pref):
    if os.path.exists(pref): return pref
    print(f"[SETUP] {pref} not found - searching /kaggle/input ...")
    hits = []
    for ext in ("*.mp4","*.avi","*.mkv","*.mov","*.MP4"):
        hits.extend(Path("/kaggle/input").rglob(ext))
    hits = sorted(hits, key=lambda p: p.stat().st_size, reverse=True)
    if not hits:
        raise SystemExit("[FATAL] no video found under /kaggle/input")
    print(f"[SETUP] using {hits[0]}")
    return str(hits[0])


def feet_in_polys(bbox, polys):
    if not polys: return False
    fx, fy = (bbox[0]+bbox[2])/2.0, bbox[3]
    return any(cv2.pointPolygonTest(p,(float(fx),float(fy)),False) >= 0 for p in polys)


def box_overlap_frac(bbox, polys):
    if not polys: return 0.0
    x1,y1,x2,y2 = (float(v) for v in bbox)
    if x2 <= x1 or y2 <= y1: return 0.0
    inside = total = 0
    for px in np.linspace(x1,x2,6):
        for py in np.linspace(y1,y2,6):
            total += 1
            if any(cv2.pointPolygonTest(p,(px,py),False) >= 0 for p in polys):
                inside += 1
    return inside/total if total else 0.0


def torso_white_ratio(frame, bbox):
    x1,y1,x2,y2 = (int(v) for v in bbox)
    h,w = y2-y1, x2-x1
    if h <= 0 or w <= 0: return 0.0
    ty1, ty2 = max(0,y1+int(0.15*h)), max(0,y1+int(0.45*h))
    tx1, tx2 = max(0,x1+int(0.20*w)), max(0,x2-int(0.20*w))
    crop = frame[ty1:ty2, tx1:tx2]
    if crop.size == 0: return 0.0
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    return float(((hsv[:,:,1] <= WHITE_S_MAX) & (hsv[:,:,2] >= WHITE_V_MIN)).mean())


def uniform_color_ratio(frame, bbox):
    x1,y1,x2,y2 = (int(v) for v in bbox)
    h,w = y2-y1, x2-x1
    if h <= 0 or w <= 0: return 0.0
    ly1, ly2 = max(0,y1+int(0.55*h)), max(0,y1+int(0.88*h))
    lx1, lx2 = max(0,x1+int(0.25*w)), max(0,x2-int(0.25*w))
    crop = frame[ly1:ly2, lx1:lx2]
    if crop.size == 0: return 0.0
    px = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV).reshape(-1,3).astype(np.int16)
    return float(np.all((px >= UNIFORM_BAND_LO) & (px <= UNIFORM_BAND_HI), axis=1).mean())


def quality_ok(bbox, shape):
    x1,y1,x2,y2 = bbox
    w,h = x2-x1, y2-y1
    if w < MIN_BOX_W or h < MIN_BOX_H: return False
    ar = h/max(w,1e-6)
    if ar < MIN_ASPECT or ar > MAX_ASPECT: return False
    H,W = shape[:2]
    if x1 <= 2 or y1 <= 2 or x2 >= W-2 or y2 >= H-2: return False
    return True


def cut_crop(frame, bbox):
    x1,y1,x2,y2 = bbox
    w,h = x2-x1, y2-y1
    px,py = CROP_PAD*w, CROP_PAD*h
    H,W = frame.shape[:2]
    a,b = int(max(0,x1-px)), int(max(0,y1-py))
    c,d = int(min(W,x2+px)), int(min(H,y2+py))
    crop = frame[b:d, a:c]
    if crop.size == 0: return None
    return cv2.resize(crop, CROP_SIZE, interpolation=cv2.INTER_AREA)


def make_contact_sheet(folder, dest, cols=12, rows=6):
    files = sorted(Path(folder).glob("*.jpg"))
    if not files: return
    picks = random.Random(0).sample(files, min(cols*rows, len(files)))
    cw, ch = 96, 192
    sheet = np.full((rows*ch, cols*cw, 3), 30, np.uint8)
    for i,p in enumerate(picks):
        img = cv2.imread(str(p))
        if img is None: continue
        r,c = divmod(i, cols)
        sheet[r*ch:(r+1)*ch, c*cw:(c+1)*cw] = cv2.resize(img, (cw,ch))
    cv2.imwrite(str(dest), sheet)
    print(f"[PREVIEW] {dest}")


def write_tracker_yaml(dest):
    import yaml, ultralytics
    src = Path(ultralytics.__file__).parent/"cfg"/"trackers"/"botsort.yaml"
    try: cfg = yaml.safe_load(src.read_text())
    except Exception: cfg = {"tracker_type":"botsort"}
    cfg.update({"tracker_type":"botsort","track_high_thresh":TRACK_HIGH_THRESH,
                "track_low_thresh":TRACK_LOW_THRESH,"new_track_thresh":NEW_TRACK_THRESH,
                "track_buffer":TRACK_BUFFER,"match_thresh":MATCH_THRESH,
                "proximity_thresh":PROXIMITY_THRESH,"appearance_thresh":APPEARANCE_THRESH,
                "with_reid":False})
    Path(dest).write_text(yaml.safe_dump(cfg))
    return str(dest)


def extract(video_path, out):
    import csv
    from ultralytics import YOLO
    dirs = {"seller": out/"dataset"/"seller", "client": out/"dataset"/"client",
            "review": out/"review"}
    for d in dirs.values(): d.mkdir(parents=True, exist_ok=True)

    model = YOLO(MODEL_NAME); model.to(DEVICE)
    tracker_yaml = write_tracker_yaml(out/"botsort_crk.yaml")

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    limit = int(MAX_MINUTES*60*fps) if MAX_MINUTES else 0
    tag = Path(video_path).stem.replace(" ","_")[:24]

    print(f"[VIDEO] {video_path}")
    print(f"        {W}x{H} @ {fps:.1f}fps, {total} frames ({total/fps/60:.1f} min), every {SKIP}th")
    build_polys(W, H)

    counts = {"seller":0,"client":0,"review":0}
    per_track = defaultdict(int); last_frame = defaultdict(lambda: -10**9)
    fields = ["file","label","reason","video","track_id","frame","time_s","det_conf",
              "color_ratio","white_ratio","seller_zone_frac","x1","y1","x2","y2"]
    fh = open(out/"manifest.csv","w",newline="")
    writer = csv.DictWriter(fh, fieldnames=fields); writer.writeheader()

    idx = -1
    while True:
        ok, frame = cap.read()
        if not ok: break
        idx += 1
        if limit and idx > limit: break
        if idx % SKIP != 0: continue

        res = model.track(frame, persist=True, tracker=tracker_yaml, conf=DETECT_CONF,
                          classes=[PERSON_CLASS_ID], iou=0.5, imgsz=640,
                          device=DEVICE, half=HALF, verbose=False)
        boxes = res[0].boxes
        if boxes is None or len(boxes) == 0 or boxes.id is None: continue

        xyxy = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        tids = boxes.id.cpu().numpy().astype(int)

        for bbox, cf, tid in zip(xyxy, confs, tids):
            if feet_in_polys(bbox, _MIRROR_POLYS): continue
            if feet_in_polys(bbox, _IGNORE_POLYS): continue
            if not quality_ok(bbox, frame.shape): continue
            if per_track[tid] >= MAX_PER_TRACK: continue
            if idx - last_frame[tid] < MIN_FRAME_GAP: continue

            color = uniform_color_ratio(frame, bbox)
            white = torso_white_ratio(frame, bbox)
            zfrac = box_overlap_frac(bbox, _SELLER_POLYS)

            if zfrac >= SELLER_ZONE_FRAC:
                label, reason = "seller", "zone"
            elif color >= SELLER_COLOR_MIN and white >= SELLER_WHITE_MIN:
                label, reason = "seller", "hsv"
            elif color <= CLIENT_COLOR_MAX and white <= CLIENT_WHITE_MAX:
                label, reason = "client", "hsv"
            else:
                label, reason = "review", "ambiguous"

            if counts[label] >= (MAX_REVIEW if label == "review" else MAX_PER_CLASS):
                continue
            crop = cut_crop(frame, bbox)
            if crop is None: continue

            name = f"{tag}_t{int(tid):05d}_f{idx:07d}.jpg"
            cv2.imwrite(str(dirs[label]/name), crop, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
            counts[label] += 1; per_track[tid] += 1; last_frame[tid] = idx
            writer.writerow({"file":f"{label}/{name}","label":label,"reason":reason,
                "video":tag,"track_id":int(tid),"frame":idx,"time_s":round(idx/fps,2),
                "det_conf":round(float(cf),3),"color_ratio":round(color,4),
                "white_ratio":round(white,4),"seller_zone_frac":round(zfrac,3),
                "x1":int(bbox[0]),"y1":int(bbox[1]),"x2":int(bbox[2]),"y2":int(bbox[3])})

        if idx % (SKIP*100) == 0:
            pct = 100.0*idx/total if total else 0.0
            print(f"  {pct:5.1f}%  seller={counts['seller']} client={counts['client']} "
                  f"review={counts['review']}", flush=True)

    cap.release(); fh.close(); del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    print("\n" + "="*60)
    print(f"seller : {counts['seller']:5d}  ->  {dirs['seller']}")
    print(f"client : {counts['client']:5d}  ->  {dirs['client']}")
    print(f"review : {counts['review']:5d}  ->  {dirs['review']}")
    print("="*60)
    for k in ("seller","client","review"):
        make_contact_sheet(dirs[k], out/f"preview_{k}.jpg")
    if counts["seller"] < 50 or counts["client"] < 50:
        print("\n[WARN] one class is very small. Lower SKIP or loosen the bands.")
    return counts


def train(out):
    import torch.nn as nn
    from PIL import Image
    from torch.utils.data import DataLoader, Dataset
    from torchvision import models, transforms

    IMG_H, IMG_W = CROP_SIZE[1], CROP_SIZE[0]
    MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
    train_tf = transforms.Compose([
        transforms.Resize((IMG_H,IMG_W)), transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.35,0.30,0.25,0.03),
        transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
        transforms.RandomAffine(6, translate=(0.05,0.05), scale=(0.92,1.08)),
        transforms.ToTensor(), transforms.Normalize(MEAN,STD),
        transforms.RandomErasing(p=0.25, scale=(0.02,0.12))])
    eval_tf = transforms.Compose([transforms.Resize((IMG_H,IMG_W)),
        transforms.ToTensor(), transforms.Normalize(MEAN,STD)])

    def track_key(p):
        parts = p.stem.rsplit("_f",1)
        return parts[0] if len(parts) == 2 else p.stem

    items = []
    for ci, cname in enumerate(CLASSES):
        for p in sorted((out/"dataset"/cname).glob("*.jpg")):
            items.append((p, ci, track_key(p)))
    if len(items) < 40:
        print(f"[SKIP TRAIN] only {len(items)} images. Lower SKIP and rerun."); return

    by_track = defaultdict(list)
    for it in items: by_track[it[2]].append(it)
    tracks = sorted(by_track); random.Random(SEED).shuffle(tracks)
    n_val = max(1, int(len(tracks)*VAL_FRAC)); val_tracks = set(tracks[:n_val])
    tr = [i for t in tracks if t not in val_tracks for i in by_track[t]]
    va = [i for t in val_tracks for i in by_track[t]]
    if not tr or not va:
        print("[SKIP TRAIN] not enough distinct tracks."); return
    if len({y for _,y,_ in tr}) < 2 or len({y for _,y,_ in va}) < 2:
        print("[SKIP TRAIN] a split ended up single-class. Extract more data."); return

    print(f"\ntotal={len(items)}  train={len(tr)}  val={len(va)}  "
          f"(tracks {len(tracks)-n_val}/{n_val})")

    class DS(Dataset):
        def __init__(s, its, tf): s.its, s.tf = its, tf
        def __len__(s): return len(s.its)
        def __getitem__(s, i):
            p,y,_ = s.its[i]
            return s.tf(Image.open(p).convert("RGB")), y

    def report(yt, yp, tag):
        yt, yp = np.array(yt), np.array(yp)
        acc = (yt == yp).mean()
        print(f"\n--- {tag} ---\naccuracy: {acc*100:.2f}%  (n={len(yt)})")
        print("            pred_client  pred_seller")
        for ci,cn in enumerate(CLASSES):
            r = [int(((yt==ci)&(yp==j)).sum()) for j in range(2)]
            print(f"true_{cn:<7}{r[0]:>10}{r[1]:>13}")
        for ci,cn in enumerate(CLASSES):
            tp = int(((yt==ci)&(yp==ci)).sum()); fp = int(((yt!=ci)&(yp==ci)).sum())
            fn = int(((yt==ci)&(yp!=ci)).sum())
            P, R = tp/max(tp+fp,1), tp/max(tp+fn,1)
            print(f"{cn:<8} P={P:.3f}  R={R:.3f}  F1={2*P*R/max(P+R,1e-9):.3f}")
        return acc

    print("\n" + "="*60 + "\nSTAGE 2a - k-NN on frozen features (no training)\n" + "="*60)
    base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    bb = nn.Sequential(*list(base.children())[:-1]).to(DEVICE).eval()

    @torch.no_grad()
    def embed(its):
        dl = DataLoader(DS(its, eval_tf), batch_size=BATCH_SIZE, num_workers=2)
        F, Y = [], []
        for x,y in dl:
            f = bb(x.to(DEVICE)).flatten(1)
            F.append(nn.functional.normalize(f, dim=1).cpu()); Y.append(y)
        return torch.cat(F), torch.cat(Y)

    ftr, ytr = embed(tr); fva, yva = embed(va)
    k = min(15, ftr.shape[0])
    votes = ytr[(fva @ ftr.T).topk(k, dim=1).indices]
    knn_acc = report(yva.tolist(), (votes.float().mean(1) > 0.5).long().tolist(), f"k-NN (k={k})")
    del bb
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    print("\n" + "="*60 + "\nSTAGE 2b - fine-tuning ResNet18\n" + "="*60)
    dl_tr = DataLoader(DS(tr, train_tf), batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=2, pin_memory=True)
    dl_va = DataLoader(DS(va, eval_tf), batch_size=BATCH_SIZE, num_workers=2)

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 2); model = model.to(DEVICE)

    nc = sum(1 for _,y,_ in tr if y == 0); ns = sum(1 for _,y,_ in tr if y == 1)
    w = torch.tensor([len(tr)/(2*max(nc,1)), len(tr)/(2*max(ns,1))],
                     dtype=torch.float, device=DEVICE)
    print(f"train client={nc} seller={ns}  weights={[round(v,2) for v in w.tolist()]}")

    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR*3,
                steps_per_epoch=max(len(dl_tr),1), epochs=EPOCHS)
    use_cuda = torch.cuda.is_available()
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)
        autocast = lambda: torch.amp.autocast("cuda", enabled=use_cuda)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=use_cuda)
        autocast = lambda: torch.cuda.amp.autocast(enabled=use_cuda)

    @torch.no_grad()
    def ev():
        model.eval(); yt, yp = [], []
        for x,y in dl_va:
            yp += model(x.to(DEVICE)).argmax(1).cpu().tolist(); yt += y.tolist()
        return yt, yp

    best, ckpt = 0.0, out/"seller_client_resnet18.pt"
    for ep in range(1, EPOCHS+1):
        model.train(); tot = corr = 0; run = 0.0
        for x,y in dl_tr:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast():
                o = model(x); loss = crit(o, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()*y.size(0); corr += (o.argmax(1) == y).sum().item(); tot += y.size(0)
        yt, yp = ev()
        vacc = (np.array(yt) == np.array(yp)).mean()
        print(f"ep {ep:02d}  loss={run/max(tot,1):.4f}  train={corr/max(tot,1)*100:.2f}%  "
              f"val={vacc*100:.2f}%")
        if vacc > best:
            best = vacc
            torch.save({"state_dict":model.state_dict(),"classes":CLASSES,
                        "img_size":[IMG_H,IMG_W],"mean":MEAN,"std":STD}, ckpt)

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["state_dict"])
    yt, yp = ev(); report(yt, yp, "fine-tuned ResNet18 (best checkpoint)")

    try:
        model.eval()
        torch.onnx.export(model, torch.randn(1,3,IMG_H,IMG_W, device=DEVICE),
            str(out/"seller_client_resnet18.onnx"), input_names=["input"],
            output_names=["logits"],
            dynamic_axes={"input":{0:"batch"},"logits":{0:"batch"}}, opset_version=13)
        print(f"[ONNX] {out/'seller_client_resnet18.onnx'}")
    except Exception as e:
        print(f"[ONNX] export skipped: {e}")

    (out/"results.json").write_text(json.dumps({"knn_acc":float(knn_acc),
        "finetune_val_acc":float(best),"n_train":len(tr),"n_val":len(va),
        "classes":CLASSES}, indent=2))
    print(f"\nk-NN {knn_acc*100:.2f}%   vs   fine-tuned {best*100:.2f}%")
    print(f"[SAVED] {ckpt}")


def main():
    print("="*60)
    print(f"[DEVICE] {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
    print(f"[NUMPY ] {np.__version__}    [TORCH] {torch.__version__}")
    print("="*60)
    video = find_video(VIDEO_PATH)
    out = Path(OUT_DIR)
    if RUN_EXTRACT:
        if out.exists(): shutil.rmtree(out)
        out.mkdir(parents=True, exist_ok=True)
        extract(video, out)
    if RUN_TRAIN:
        train(out)
    try:
        shutil.make_archive(str(out.parent/"crk_dataset"), "zip", root_dir=str(out))
        print(f"[ZIP] {out.parent/'crk_dataset'}.zip")
    except Exception as e:
        print(f"[ZIP] skipped: {e}")



In [ ]:
# ===== v2: track-level labelling, single video pass =====
SKIP = 10
MAX_PER_TRACK = 60
MAX_PER_CLASS = 6000
MAX_REVIEW = 600

# thresholds derived from YOUR zone-labelled sellers
TRK_SELLER_WHITE = 0.55
TRK_SELLER_COLOR = 0.18
TRK_CLIENT_WHITE = 0.28
TRK_CLIENT_COLOR = 0.12

def extract(video_path, out):
    import csv
    from ultralytics import YOLO

    pool = out / "_pool"; pool.mkdir(parents=True, exist_ok=True)
    dirs = {"seller": out/"dataset"/"seller", "client": out/"dataset"/"client",
            "review": out/"review"}
    for d in dirs.values(): d.mkdir(parents=True, exist_ok=True)

    model = YOLO(MODEL_NAME); model.to(DEVICE)
    tracker_yaml = write_tracker_yaml(out/"botsort_crk.yaml")

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    tag = Path(video_path).stem.replace(" ","_")[:24]
    print(f"[VIDEO] {W}x{H} @ {fps:.1f}fps, {total/fps/60:.1f} min, every {SKIP}th")
    build_polys(W, H)

    trk = defaultdict(lambda: {"white": [], "color": [], "zone": 0.0, "files": []})
    per_track = defaultdict(int); last_frame = defaultdict(lambda: -10**9)
    rows = []
    idx = -1

    while True:
        ok, frame = cap.read()
        if not ok: break
        idx += 1
        if idx % SKIP != 0: continue

        res = model.track(frame, persist=True, tracker=tracker_yaml, conf=DETECT_CONF,
                          classes=[PERSON_CLASS_ID], iou=0.5, imgsz=640,
                          device=DEVICE, half=HALF, verbose=False)
        b = res[0].boxes
        if b is None or len(b) == 0 or b.id is None: continue

        for bbox, cf, tid in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy(),
                                 b.id.cpu().numpy().astype(int)):
            if feet_in_polys(bbox, _MIRROR_POLYS): continue
            if feet_in_polys(bbox, _IGNORE_POLYS): continue
            if not quality_ok(bbox, frame.shape): continue

            color = uniform_color_ratio(frame, bbox)
            white = torso_white_ratio(frame, bbox)
            zfrac = box_overlap_frac(bbox, _SELLER_POLYS)

            t = trk[tid]
            t["white"].append(white); t["color"].append(color)
            t["zone"] = max(t["zone"], zfrac)

            if per_track[tid] >= MAX_PER_TRACK: continue
            if idx - last_frame[tid] < MIN_FRAME_GAP: continue
            crop = cut_crop(frame, bbox)
            if crop is None: continue

            name = f"{tag}_t{int(tid):05d}_f{idx:07d}.jpg"
            cv2.imwrite(str(pool/name), crop, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
            t["files"].append(name)
            per_track[tid] += 1; last_frame[tid] = idx
            rows.append({"file": name, "track_id": int(tid), "frame": idx,
                         "time_s": round(idx/fps, 2), "det_conf": round(float(cf), 3),
                         "color_ratio": round(color, 4), "white_ratio": round(white, 4),
                         "seller_zone_frac": round(zfrac, 3)})

        if idx % (SKIP*200) == 0:
            print(f"  {100.0*idx/max(total,1):5.1f}%  tracks={len(trk)} "
                  f"pooled={len(rows)}", flush=True)

    cap.release(); del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ---- decide ONE label per track ----
    label_of, why = {}, {}
    for tid, t in trk.items():
        if not t["files"]: continue
        mw = float(np.median(t["white"])); mc = float(np.median(t["color"]))
        if t["zone"] >= SELLER_ZONE_FRAC:
            label_of[tid], why[tid] = "seller", "zone"
        elif mw >= TRK_SELLER_WHITE and mc >= TRK_SELLER_COLOR:
            label_of[tid], why[tid] = "seller", "hsv"
        elif mw <= TRK_CLIENT_WHITE and mc <= TRK_CLIENT_COLOR:
            label_of[tid], why[tid] = "client", "hsv"
        else:
            label_of[tid], why[tid] = "review", "ambiguous"

    counts = {"seller":0, "client":0, "review":0}
    fmap = {}
    for tid, t in trk.items():
        lab = label_of.get(tid)
        if lab is None: continue
        for n in t["files"]:
            cap_for = MAX_REVIEW if lab == "review" else MAX_PER_CLASS
            if counts[lab] >= cap_for: break
            src = pool/n
            if not src.exists(): continue
            shutil.move(str(src), str(dirs[lab]/n))
            fmap[n] = (lab, why[tid]); counts[lab] += 1

    shutil.rmtree(pool, ignore_errors=True)

    with open(out/"manifest.csv", "w", newline="") as fh:
        wtr = csv.DictWriter(fh, fieldnames=["file","label","reason","track_id","frame",
            "time_s","det_conf","color_ratio","white_ratio","seller_zone_frac"])
        wtr.writeheader()
        for r in rows:
            if r["file"] not in fmap: continue
            lab, rsn = fmap[r["file"]]
            r = dict(r); r["label"], r["reason"] = lab, rsn
            r["file"] = f"{lab}/{r['file']}"
            wtr.writerow(r)

    ntrk = {k: sum(1 for v in label_of.values() if v == k) for k in counts}
    print("\n" + "="*60)
    for k in ("seller","client","review"):
        print(f"{k:7s}: {counts[k]:5d} crops  from {ntrk[k]:4d} tracks")
    print("="*60)
    for k in ("seller","client","review"):
        make_contact_sheet(dirs[k], out/f"preview_{k}.jpg")
    return counts

main()

In [ ]:
# ===== v3: high-yield, multi-video =====
SKIP = 3                 # was 10 -> ~3.3x more frames per track
MIN_FRAME_GAP = 3
MAX_PER_TRACK = 300      # was 60
MAX_PER_CLASS = 20000
MAX_REVIEW = 1500

TRK_SELLER_WHITE, TRK_SELLER_COLOR = 0.55, 0.18
TRK_CLIENT_WHITE, TRK_CLIENT_COLOR = 0.30, 0.14

def find_all_videos():
    hits = []
    for ext in ("*.mp4","*.avi","*.mkv","*.mov","*.MP4","*.MKV"):
        hits.extend(Path("/kaggle/input").rglob(ext))
    return sorted(set(hits), key=lambda p: p.stat().st_size, reverse=True)


def extract_many(out):
    import csv
    from ultralytics import YOLO

    videos = find_all_videos()
    if not videos:
        raise SystemExit("[FATAL] no videos under /kaggle/input")
    print(f"[SETUP] {len(videos)} video(s) found:")
    for v in videos:
        print(f"        {v}  ({v.stat().st_size/1e9:.2f} GB)")

    pool = out/"_pool"; pool.mkdir(parents=True, exist_ok=True)
    dirs = {"seller": out/"dataset"/"seller", "client": out/"dataset"/"client",
            "review": out/"review"}
    for d in dirs.values(): d.mkdir(parents=True, exist_ok=True)

    model = YOLO(MODEL_NAME); model.to(DEVICE)
    tracker_yaml = write_tracker_yaml(out/"botsort_crk.yaml")

    trk = {}
    rows = []

    for vi, vpath in enumerate(videos, 1):
        cap = cv2.VideoCapture(str(vpath))
        if not cap.isOpened():
            print(f"[SKIP] cannot open {vpath}"); continue
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        tag = f"v{vi:02d}_" + vpath.stem.replace(" ","_")[:18]
        print(f"\n[VIDEO {vi}/{len(videos)}] {vpath.name}")
        print(f"   {W}x{H} @ {fps:.1f}fps, {total/fps/60:.1f} min, every {SKIP}th frame")
        build_polys(W, H)

        model.predictor = None          # reset tracker state between videos
        per_track = defaultdict(int)
        last_frame = defaultdict(lambda: -10**9)
        idx, bad = -1, 0

        while True:
            ok, frame = cap.read()
            if not ok:
                bad += 1; idx += 1
                if bad > 300: break
                continue
            bad = 0
            idx += 1
            if idx % SKIP != 0: continue

            res = model.track(frame, persist=True, tracker=tracker_yaml,
                              conf=DETECT_CONF, classes=[PERSON_CLASS_ID], iou=0.5,
                              imgsz=640, device=DEVICE, verbose=False)
            b = res[0].boxes
            if b is None or len(b) == 0 or b.id is None: continue

            for bbox, cf, rawid in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy(),
                                       b.id.cpu().numpy().astype(int)):
                if feet_in_polys(bbox, _MIRROR_POLYS): continue
                if feet_in_polys(bbox, _IGNORE_POLYS): continue
                if not quality_ok(bbox, frame.shape): continue

                gid = f"{tag}_t{int(rawid):05d}"      # globally unique track key
                color = uniform_color_ratio(frame, bbox)
                white = torso_white_ratio(frame, bbox)
                zfrac = box_overlap_frac(bbox, _SELLER_POLYS)

                t = trk.setdefault(gid, {"white":[], "color":[], "zone":0.0, "files":[]})
                t["white"].append(white); t["color"].append(color)
                t["zone"] = max(t["zone"], zfrac)

                if per_track[gid] >= MAX_PER_TRACK: continue
                if idx - last_frame[gid] < MIN_FRAME_GAP: continue
                crop = cut_crop(frame, bbox)
                if crop is None: continue

                name = f"{gid}_f{idx:07d}.jpg"
                cv2.imwrite(str(pool/name), crop, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
                t["files"].append(name)
                per_track[gid] += 1; last_frame[gid] = idx
                rows.append({"file":name, "track_id":gid, "frame":idx,
                             "time_s":round(idx/fps,2), "det_conf":round(float(cf),3),
                             "color_ratio":round(color,4), "white_ratio":round(white,4),
                             "seller_zone_frac":round(zfrac,3)})

            if idx % (SKIP*500) == 0:
                print(f"   {100.0*idx/max(total,1):5.1f}%  tracks={len(trk)} "
                      f"pooled={len(rows)}", flush=True)
        cap.release()

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ---- one label per track ----
    label_of, why = {}, {}
    for gid, t in trk.items():
        if not t["files"]: continue
        mw, mc = float(np.median(t["white"])), float(np.median(t["color"]))
        if t["zone"] >= SELLER_ZONE_FRAC:
            label_of[gid], why[gid] = "seller", "zone"
        elif mw >= TRK_SELLER_WHITE and mc >= TRK_SELLER_COLOR:
            label_of[gid], why[gid] = "seller", "hsv"
        elif mw <= TRK_CLIENT_WHITE and mc <= TRK_CLIENT_COLOR:
            label_of[gid], why[gid] = "client", "hsv"
        else:
            label_of[gid], why[gid] = "review", "ambiguous"

    # ---- interleave tracks so caps don't starve late videos ----
    counts = {"seller":0, "client":0, "review":0}
    fmap = {}
    order = defaultdict(list)
    for gid, lab in label_of.items():
        order[lab].append(gid)
    for lab in order:
        random.Random(SEED).shuffle(order[lab])

    for lab, gids in order.items():
        cap_for = MAX_REVIEW if lab == "review" else MAX_PER_CLASS
        pos, alive = 0, list(gids)
        while alive and counts[lab] < cap_for:
            nxt = []
            for gid in alive:
                fl = trk[gid]["files"]
                if pos >= len(fl): continue
                if counts[lab] >= cap_for: break
                src = pool/fl[pos]
                if src.exists():
                    shutil.move(str(src), str(dirs[lab]/fl[pos]))
                    fmap[fl[pos]] = (lab, why[gid]); counts[lab] += 1
                nxt.append(gid)
            alive = nxt; pos += 1

    shutil.rmtree(pool, ignore_errors=True)

    with open(out/"manifest.csv","w",newline="") as fh:
        wtr = csv.DictWriter(fh, fieldnames=["file","label","reason","track_id","frame",
              "time_s","det_conf","color_ratio","white_ratio","seller_zone_frac"])
        wtr.writeheader()
        for r in rows:
            if r["file"] not in fmap: continue
            lab, rsn = fmap[r["file"]]
            r = dict(r); r["label"], r["reason"] = lab, rsn
            r["file"] = f"{lab}/{r['file']}"
            wtr.writerow(r)

    ntrk = {k: sum(1 for v in label_of.values() if v == k) for k in counts}
    print("\n" + "="*60)
    for k in ("seller","client","review"):
        print(f"{k:7s}: {counts[k]:6d} crops  from {ntrk[k]:4d} tracks")
    print("="*60)
    for k in ("seller","client","review"):
        make_contact_sheet(dirs[k], out/f"preview_{k}.jpg")
    return counts


out = Path(OUT_DIR)
if out.exists(): shutil.rmtree(out)
out.mkdir(parents=True, exist_ok=True)
extract_many(out)
train(out)
try:
    shutil.make_archive(str(out.parent/"crk_dataset"), "zip", root_dir=str(out))
    print(f"[ZIP] {out.parent/'crk_dataset'}.zip")
except Exception as e:
    print(f"[ZIP] skipped: {e}")